# 1.6 Walking-time matrices from HDB blocks to childcare centres (OSRM)

Accessibility is measured as walking time on the real footpath network, not straight-line distance.
A local OSRM server (foot profile, Singapore clip of the Geofabrik extract; setup in the README)
answers `/table` requests, and for each destination set the full origin × destination matrix of
walking minutes and metres is fetched in 100 × 100 chunks, labelled by origin `OBJECTID` and
destination id, and stored as parquet in the git-ignored `large_data/`. Origins are the 10,769
individual residential blocks (not the clusters, decided 2026-10-26). Destination sets: the
canonical 2026 centres (1,796), ECDA's 13 planned centres, the 2021 centres (comparison only) and
MRT exits (613). The per-block summary that the rest of the pipeline uses, `output/hdb_walk_access.csv`
(nearest minutes / metres / id and counts within 10 and 15 minutes per set), is rebuilt here from
the parquet files. Original scripts: `scripts/build_walk_matrix.py`, `scripts/summarise_walk_matrix.py`
(Minh); `osrm_table`, `load_dest` and the destination registry now live in `src/bt4015/osrm.py`.

**The matrix build is not re-run by default.** The centres matrix alone is 10,769 × 1,796 cells
(about 1.7 h of `/table` calls). `REBUILD_MATRIX = False` keeps the saved parquet files (built
2026-10-28, centres and planned rebuilt 2026-11-01 on the canonical layer); set it to `True` with
OSRM running on `http://localhost:5001` to rebuild the sets in `SETS_TO_BUILD`.

**Bus stops are deliberately not a destination set.** Every block is within one to six minutes of a
stop, so the variable has no spread (prof-confirmed 2026-10-30, `docs/decisions.md`); the evidence
is in [`../0_Data_Exploration/bus_stop_variance_check.ipynb`](../0_Data_Exploration/bus_stop_variance_check.ipynb).

**Inputs**
- `output/hdb_blocks.gpkg`, layer `hdb_points` (1_03) — origins.
- `output/childcare_centres_2026.gpkg`, layers `centres` (1_05) and `planned` (1_06);
  `data/ChildCareServices_2021.geojson`; `data/MRTStationExits_2026.geojson`.
- `large_data/walk_<set>_time_min.parquet`, `large_data/walk_<set>_dist_m.parquet` — saved matrices.

**Outputs**
- `large_data/walk_<set>_{time_min,dist_m}.parquet` — only when `REBUILD_MATRIX = True`.
- `output/hdb_walk_access.csv` — one row per block (10,769): nearest walk minutes / metres / id and
  counts within 10 and 15 min for `centres`, `centres2021`, `mrt_exits`, `planned`; read by 1_08
  and the pressure-index notebooks.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyogrio
import pyarrow.parquet as pq
from tqdm import tqdm

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

from bt4015.osrm import CHUNK, THRESHOLDS_MIN, DEST_SETS, load_dest, lonlat, osrm_table, osrm_reachable

GPKG = OUT / "hdb_blocks.gpkg"
ACCESS_CSV = OUT / "hdb_walk_access.csv"
OSRM_BASE = "http://localhost:5001"          # osrm-routed --algorithm mld --max-table-size 10000 -p 5001
REBUILD_MATRIX = False                       # True re-queries OSRM for SETS_TO_BUILD (~1.7 h for centres)
SETS_TO_BUILD = ("centres", "planned")       # sets rebuilt when REBUILD_MATRIX is True

## 1. Origins and the OSRM server

The origin ids are the blocks' `OBJECTID`, which is what every matrix row is labelled with, so a
matrix can be joined back to blocks regardless of row order.

In [2]:
# 1. Residential block centroids (origins) in lon/lat for OSRM.
origins = pyogrio.read_dataframe(GPKG, layer="hdb_points")
o_xy, o_ids = lonlat(origins), origins["OBJECTID"].to_numpy()
assert len(set(o_ids)) == len(o_ids), "origin ids are not unique"
print(f"{len(origins):,} origins (residential HDB blocks)")

# 2. Is a server answering? Only needed when rebuilding.
osrm_up = osrm_reachable(OSRM_BASE)
print(f"OSRM at {OSRM_BASE}: {'reachable' if osrm_up else 'not reachable'}; REBUILD_MATRIX = {REBUILD_MATRIX}")

10,769 origins (residential HDB blocks)
OSRM at http://localhost:5001: reachable; REBUILD_MATRIX = False


## 2. Build the matrices (skipped unless `REBUILD_MATRIX`)

For each destination set, two `/table` calls per 100 × 100 chunk (durations, then distances).
Unreachable pairs come back as `null` and are kept as NaN, then counted: a handful of destinations
(two centres and the MRT exit inside Changi Airport) are cut off from the public footpath network
and are expected to be unreachable from every block.

In [3]:
def build(base, name, o_xy, o_ids):
    """Fetch the full origin x destination matrix for DEST_SETS[name] and save it as parquet.

    Returns the per-origin summary (nearest minutes / id / metres, counts within THRESHOLDS_MIN).
    """
    dest, d_ids = load_dest(name)
    d_xy = lonlat(dest)
    T = np.full((len(o_xy), len(d_xy)), np.nan)
    D = np.full_like(T, np.nan)
    o_chunks = range(0, len(o_xy), CHUNK)
    d_chunks = range(0, len(d_xy), CHUNK)
    with tqdm(total=len(o_chunks) * len(d_chunks), desc=name, unit="call") as bar:
        for i in o_chunks:
            for j in d_chunks:
                so, sd = o_xy[i:i + CHUNK], d_xy[j:j + CHUNK]
                T[i:i + CHUNK, j:j + CHUNK] = osrm_table(base, so, sd, "duration") / 60.0
                D[i:i + CHUNK, j:j + CHUNK] = osrm_table(base, so, sd, "distance")
                bar.update(1)
    n_nan = int(np.isnan(T).sum())
    print(f"{name}: {T.shape[0]:,} x {T.shape[1]:,}; unreachable cells: {n_nan:,} "
          f"({n_nan / T.size:.4%}); median nearest = {np.nanmedian(np.nanmin(T, axis=1)):.1f} min")
    LARGE.mkdir(exist_ok=True)
    for arr, tag in ((T, "time_min"), (D, "dist_m")):
        df = pd.DataFrame(arr, index=pd.Index(o_ids, name="OBJECTID"), columns=d_ids)
        df.to_parquet(LARGE / f"walk_{name}_{tag}.parquet")
    summary = pd.DataFrame({"OBJECTID": o_ids,
                            f"{name}_nearest_min": np.nanmin(T, axis=1),
                            f"{name}_nearest_id": d_ids[np.nanargmin(np.where(np.isnan(T), np.inf, T), axis=1)],
                            f"{name}_nearest_m": D[np.arange(len(T)), np.nanargmin(np.where(np.isnan(T), np.inf, T), axis=1)]})
    for t in THRESHOLDS_MIN:
        summary[f"{name}_n_within_{t}min"] = (T <= t).sum(axis=1)
    return summary


if REBUILD_MATRIX:
    # 1. Rebuild only with a live server; a failed call raises rather than writing a partial matrix.
    assert osrm_up, f"OSRM not reachable on {OSRM_BASE}; start osrm-routed first"
    for name in SETS_TO_BUILD:
        build(OSRM_BASE, name, o_xy, o_ids)
else:
    # 2. Report what is on disk instead (parquet metadata only, nothing is loaded).
    print("Using the saved matrices in large_data/ (REBUILD_MATRIX = False):")
    for p in sorted(LARGE.glob("walk_*_time_min.parquet")):
        meta = pq.read_metadata(p)
        print(f"  {p.name:36s} {meta.num_rows:,} origins x {meta.num_columns:,} destinations")

Using the saved matrices in large_data/ (REBUILD_MATRIX = False):
  walk_centres2021_time_min.parquet    10,769 origins x 1,926 destinations
  walk_centres_time_min.parquet        10,769 origins x 1,797 destinations
  walk_mrt_exits_time_min.parquet      10,769 origins x 614 destinations
  walk_planned_time_min.parquet        10,769 origins x 14 destinations


## 3. Per-block summary from the parquet matrices → `hdb_walk_access.csv`

Every set with both a time and a distance matrix in `large_data/` is summarised: nearest destination
(minutes, id, metres) and the number of destinations within 10 and 15 minutes. An origin with no
reachable destination gets `inf` for its nearest time.

In [4]:
def summarise(name):
    """Per-origin summary of one destination set from its saved parquet matrices."""
    T = pd.read_parquet(LARGE / f"walk_{name}_time_min.parquet")
    D = pd.read_parquet(LARGE / f"walk_{name}_dist_m.parquet")
    t = T.to_numpy(float)
    safe = np.where(np.isnan(t), np.inf, t)
    j = safe.argmin(axis=1)
    s = pd.DataFrame({"OBJECTID": T.index.to_numpy(),
                      f"{name}_nearest_min": safe[np.arange(len(t)), j],
                      f"{name}_nearest_id": T.columns.to_numpy()[j],
                      f"{name}_nearest_m": D.to_numpy(float)[np.arange(len(t)), j]})
    for th in THRESHOLDS_MIN:
        s[f"{name}_n_within_{th}min"] = (t <= th).sum(axis=1)
    unreach = np.isinf(s[f"{name}_nearest_min"]).sum()
    print(f"{name}: {T.shape[0]:,} x {T.shape[1]:,}; nan cells {int(np.isnan(t).sum()):,}; "
          f"origins with no reachable destination {unreach}; "
          f"median nearest {np.nanmedian(t.min(axis=1, initial=np.inf, where=~np.isnan(t))):.1f} min")
    return s


# 1. Block identifiers and context columns, then one summary block per destination set.
summary = origins[["OBJECTID", "cluster_id", "SUBZONE_C", "SUBZONE_N", "PLN_AREA_N",
                   "total_dwelling_units"]].copy()
sets = sorted({p.name.split("_time_min")[0].replace("walk_", "")
               for p in LARGE.glob("walk_*_time_min.parquet")
               if (LARGE / p.name.replace("_time_min", "_dist_m")).exists()})
if not sets:
    raise SystemExit("no complete parquet matrices found in large_data/")
for name in sets:
    summary = summary.merge(summarise(name), on="OBJECTID", how="left")
summary.to_csv(ACCESS_CSV, index=False)
print(f"Wrote {ACCESS_CSV.relative_to(ROOT)} ({len(summary):,} rows, sets: {', '.join(sets)})")

centres: 10,769 x 1,796; nan cells 21,538; origins with no reachable destination 0; median nearest 3.1 min


centres2021: 10,769 x 1,925; nan cells 10,769; origins with no reachable destination 0; median nearest 3.1 min


mrt_exits: 10,769 x 613; nan cells 10,769; origins with no reachable destination 0; median nearest 9.2 min
planned: 10,769 x 13; nan cells 0; origins with no reachable destination 0; median nearest 45.3 min
Wrote output/hdb_walk_access.csv (10,769 rows, sets: centres, centres2021, mrt_exits, planned)


## 4. Headline numbers

Walking access to *a* centre is near-universal, which is why the later analysis turns to supply per
child and the walk-time distribution rather than a binary "has access".

In [5]:
# 1. Nearest-centre walk time across blocks, and how many blocks have no centre within 15 min.
near = summary["centres_nearest_min"].replace(np.inf, np.nan)
print(f"Nearest 2026 centre: median {near.median():.1f} min, p90 {near.quantile(.9):.1f} min, max {near.max():.1f} min")
print(f"Blocks with no centre within 10 min: {(summary['centres_n_within_10min'] == 0).sum():,}; "
      f"within 15 min: {(summary['centres_n_within_15min'] == 0).sum():,}")
print(f"Centres within 10 min per block: median {summary['centres_n_within_10min'].median():.0f}, "
      f"max {summary['centres_n_within_10min'].max()}")

# 2. MRT for contrast: a covariate with real spread (see the bus-stop note in the title cell).
mrt = summary["mrt_exits_nearest_min"].replace(np.inf, np.nan)
print(f"Nearest MRT exit: median {mrt.median():.1f} min, p90 {mrt.quantile(.9):.1f} min; "
      f"blocks more than 15 min from an exit: {(mrt > 15).sum():,}")
print(f"Nearest planned centre: median {summary['planned_nearest_min'].replace(np.inf, np.nan).median():.1f} min; "
      f"blocks with a planned centre within 10 min: {(summary['planned_n_within_10min'] >= 1).sum():,}")

Nearest 2026 centre: median 3.1 min, p90 6.0 min, max 52.9 min
Blocks with no centre within 10 min: 33; within 15 min: 3
Centres within 10 min per block: median 7, max 26
Nearest MRT exit: median 9.2 min, p90 19.6 min; blocks more than 15 min from an exit: 2,350
Nearest planned centre: median 45.3 min; blocks with a planned centre within 10 min: 579
